# 00 · Environment check
Proof that this project runs on IBM LinuxONE and uses the Telum AI accelerator.
**Screenshot the outputs of this notebook for the submission video.**

In [1]:
import platform, subprocess
print("Machine architecture:", platform.machine())   # s390x = IBM Z
features = open("/proc/cpuinfo").read().split("features")[1].split("\n")[0]
print("NNPA (on-chip AI accelerator) present:", "nnpa" in features)
print(subprocess.run(["grep", "-m1", "machine", "/proc/cpuinfo"], capture_output=True, text=True).stdout.strip())

Machine architecture: s390x
NNPA (on-chip AI accelerator) present: True
processor 0: version = FF,  identification = 0ADFD8,  machine = 9175


In [2]:
import pandas, sklearn, xgboost, torch, torch_nnpa
print("pandas", pandas.__version__, "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__, "| torch", torch.__version__)
print("Accelerator available to PyTorch:", torch.nnpa.is_available(), "| Telum II:", torch.nnpa.is_telum2_available())

pandas 3.0.6 | scikit-learn 1.9.0 | xgboost 3.4.1 | torch 2.11.0+cpu
Accelerator available to PyTorch: True | Telum II: True


## Accelerator speed test
Same small neural network, same 4,096 inputs, run on the regular CPU and on the Telum accelerator.
Each run copies the result back to the CPU so the timing is honest. Numbers vary a little because the server is shared.

In [3]:
import time, copy
torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(32, 256), torch.nn.ReLU(),
                            torch.nn.Linear(256, 256), torch.nn.ReLU(),
                            torch.nn.Linear(256, 1)).eval()
x = torch.randn(4096, 32)
with torch.no_grad():
    reference = model(x)

def time_it(fn, n=30):
    with torch.no_grad():
        for _ in range(3): fn()
        t = time.perf_counter()
        for _ in range(n): out = fn()
    return (time.perf_counter() - t) / n * 1000, out

cpu_ms, _ = time_it(lambda: model(x))
model_nnpa = copy.deepcopy(model).to("nnpa")
nnpa_ms, out = time_it(lambda: model_nnpa(x.to("nnpa")).cpu())
print(f"CPU:         {cpu_ms:6.2f} ms per 4,096 predictions")
print(f"Accelerator: {nnpa_ms:6.2f} ms per 4,096 predictions (including data copy)")
print(f"Speedup:     {cpu_ms / nnpa_ms:.1f}x")
print(f"Largest difference from CPU result: {(out - reference).abs().max().item():.4f} (reduced-precision math)")

CPU:          17.24 ms per 4,096 predictions
Accelerator:   0.45 ms per 4,096 predictions (including data copy)
Speedup:     38.6x
Largest difference from CPU result: 0.0015 (reduced-precision math)
